In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/nnetnav-wa", split="train")
dataset = dataset.remove_columns(['prompt', 'output'])

# dataset = dataset.select(range(100))

/home/jadeleiyu/miniforge3/envs/mbrl_agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset[3]['messages']

[{'role': 'system',
  'content': 'You are an autonomous intelligent agent tasked with navigating a web browser. You will be given web-based tasks. These tasks will be accomplished through the use of specific actions you can issue.\n\nHere\'s the information you\'ll have:\nThe user\'s objective: This is the task you\'re trying to complete.\nThe current web page\'s accessibility tree: This is a simplified representation of the webpage, providing key information.\nThe current web page\'s URL: This is the page you\'re currently navigating.\nThe open tabs: These are the tabs you have open.\nThe previous actions: These are all the action you have performed. It may be helpful to track your progress.\n\nThe actions you can perform fall into several categories:\n\nPage Operation Actions:\n`click [id]`: This action clicks on an element with a specific id on the webpage.\n`type [id] [content] [press_enter_after=0|1]`: Use this to type the content into the field with id. By default, the "Enter" ke

In [4]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config
from transformers import AutoTokenizer

model_name = "openai/gpt-oss-20b"

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████| 3/3 [00:21<00:00,  7.18s/it]


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    max_length=4096,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir="gpt-oss-20b-nnetnav-wa",
    # assistant_only_loss=True
)

In [10]:
dataset_test = dataset.select(range(100))

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_test,
    processing_class=tokenizer,
)
trainer.train()


KeyboardInterrupt: 